In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import joblib

In [ ]:
DATA_PATH = '../data/processed/stroke_processed.csv' 
SCALER_PATH = '../output/stroke_scaler.joblib'
MODEL_SAVE_PATH = '../output/stroke_cluster_model.joblib'

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
X = df.drop(columns=['stroke'])

In [ ]:
scaler = joblib.load(SCALER_PATH)
X_scaled = scaler.transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
inertia, silhouette_scores = [], []
K_range = range(2, 8)

In [ ]:
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled_df)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled_df, kmeans.labels_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertia, marker='o')
axes[0].set_title('Stroke: Elbow Method')
axes[1].plot(K_range, silhouette_scores, marker='o', color='purple')
axes[1].set_title('Stroke: Silhouette Score')
plt.show()

In [ ]:
OPTIMAL_K = 3 
final_kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df['Cluster'] = final_kmeans.fit_predict(X_scaled_df)

In [ ]:
cluster_centroids = df.groupby('Cluster').mean()
print("\n--- Stroke Risk Profiles ---")
display(cluster_centroids)

In [ ]:
joblib.dump(final_kmeans, MODEL_SAVE_PATH)
print(f"\n✅ Stroke cluster model saved to {MODEL_SAVE_PATH}")